# 02 — Data Cleaning

## Objective

Clean and transform the official HDB resale transaction dataset into an analysis-ready dataset while preserving the original raw data.

In [10]:
from pathlib import Path

PROJECT_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data").is_dir() and (path / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project directory.")

import pandas as pd
import numpy as np

df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "hdb_resale_raw.csv")

df.shape

(239330, 11)

### 1. Convert month to a real date

In [11]:
df["month"].dtype

dtype('O')

In [12]:
df["month"] = pd.to_datetime(
    df["month"],
    format="%Y-%m"
)

In [13]:
df["month"].min(), df["month"].max()

(Timestamp('2017-01-01 00:00:00'), Timestamp('2026-08-01 00:00:00'))

### 2. Checking missing values

In [6]:
df.isna().sum()

month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

No missing values were identified across the 11 original variables. Therefore, no imputation or row removal was required.

### 3. Investigate the 318 apparent duplicates

In [7]:
df.duplicated().sum()

np.int64(318)

In [8]:
duplicates = df[df.duplicated(keep=False)]

duplicates.sort_values(
    ["month", "town", "block", "street_name"]
).head(20)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
224,2017-01-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81.0,Improved,1975,57 years,470000.0
243,2017-01-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81.0,Improved,1975,57 years,470000.0
304,2017-01-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68.0,Improved,1979,61 years 02 months,470000.0
305,2017-01-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68.0,Improved,1979,61 years 02 months,470000.0
505,2017-01-01,JURONG EAST,4 ROOM,265,TOH GUAN RD,04 TO 06,101.0,Model A,1998,80 years 09 months,470000.0
510,2017-01-01,JURONG EAST,4 ROOM,265,TOH GUAN RD,04 TO 06,101.0,Model A,1998,80 years 09 months,470000.0
533,2017-01-01,JURONG WEST,4 ROOM,337A,TAH CHING RD,19 TO 21,92.0,Model A,2010,92 years 03 months,470000.0
591,2017-01-01,JURONG WEST,4 ROOM,337A,TAH CHING RD,19 TO 21,92.0,Model A,2010,92 years 03 months,470000.0
671,2017-01-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105.0,Model A,1996,78 years 10 months,368000.0
672,2017-01-01,PASIR RIS,4 ROOM,753,PASIR RIS ST 71,01 TO 03,105.0,Model A,1996,78 years 10 months,368000.0


### Duplicate investigation

318 rows were flagged as exact duplicates based on all available columns. However, the dataset does not provide a unique transaction identifier, and multiple transactions can share the same recorded property characteristics, transaction month, and resale price.

Therefore, exact matching rows were retained rather than automatically removed to avoid deleting potentially legitimate transactions.

### 4. Validate

In [9]:
sorted(df["town"].unique())

['ANG MO KIO',
 'BEDOK',
 'BISHAN',
 'BUKIT BATOK',
 'BUKIT MERAH',
 'BUKIT PANJANG',
 'BUKIT TIMAH',
 'CENTRAL AREA',
 'CHOA CHU KANG',
 'CLEMENTI',
 'GEYLANG',
 'HOUGANG',
 'JURONG EAST',
 'JURONG WEST',
 'KALLANG/WHAMPOA',
 'MARINE PARADE',
 'PASIR RIS',
 'PUNGGOL',
 'QUEENSTOWN',
 'SEMBAWANG',
 'SENGKANG',
 'SERANGOON',
 'TAMPINES',
 'TOA PAYOH',
 'WOODLANDS',
 'YISHUN']

In [10]:
df["flat_type"].value_counts()

flat_type
4 ROOM              101732
5 ROOM               58516
3 ROOM               56895
EXECUTIVE            16991
2 ROOM                5020
1 ROOM                  88
MULTI-GENERATION        88
Name: count, dtype: int64

In [11]:
sorted(df["flat_model"].unique())

['2-room',
 '3Gen',
 'Adjoined flat',
 'Apartment',
 'DBSS',
 'Improved',
 'Improved-Maisonette',
 'Maisonette',
 'Model A',
 'Model A-Maisonette',
 'Model A2',
 'Multi Generation',
 'New Generation',
 'Premium Apartment',
 'Premium Apartment Loft',
 'Premium Maisonette',
 'Simplified',
 'Standard',
 'Terrace',
 'Type S1',
 'Type S2']

In [12]:
sorted(df["storey_range"].unique())

['01 TO 03',
 '04 TO 06',
 '07 TO 09',
 '10 TO 12',
 '13 TO 15',
 '16 TO 18',
 '19 TO 21',
 '22 TO 24',
 '25 TO 27',
 '28 TO 30',
 '31 TO 33',
 '34 TO 36',
 '37 TO 39',
 '40 TO 42',
 '43 TO 45',
 '46 TO 48',
 '49 TO 51']

### 5. Cleaning remaining_lease

In [30]:
df["remaining_lease_years"] = (
    df["remaining_lease"]
    .str.extract(r"(\d+)\s+years?")[0]
    .astype(int)
)

In [31]:
df["remaining_lease_extra_months"] = (
    df["remaining_lease"]
    .str.extract(r"(\d+)\s+months?")[0]
    .fillna(0)
    .astype(int)
)

In [32]:
df["remaining_lease_months"] = (
    df["remaining_lease_years"] * 12
    + df["remaining_lease_extra_months"]
)

In [33]:
df[
    [
        "remaining_lease",
        "remaining_lease_years",
        "remaining_lease_extra_months",
        "remaining_lease_months"
    ]
].head(10)

,remaining_lease,remaining_lease_years,remaining_lease_extra_months,remaining_lease_months
0,61 years 04 months,61,4,736
1,60 years 07 months,60,7,727
2,62 years 05 months,62,5,749
3,62 years 01 month,62,1,745
4,62 years 05 months,62,5,749
5,63 years,63,0,756
6,61 years 06 months,61,6,738
7,58 years 04 months,58,4,700
8,61 years 06 months,61,6,738
9,61 years 04 months,61,4,736


### 6. Validate

In [17]:
df["floor_area_sqm"].describe()

count    239330.000000
mean         96.683115
std          24.013807
min          31.000000
25%          81.000000
50%          93.000000
75%         112.000000
max         366.700000
Name: floor_area_sqm, dtype: float64

In [18]:
df.nlargest(
    10,
    "floor_area_sqm"
)[
    ["town", "flat_type", "flat_model",
     "floor_area_sqm", "resale_price"]
]

,town,flat_type,flat_model,floor_area_sqm,resale_price
182823,KALLANG/WHAMPOA,3 ROOM,Terrace,366.7,1568000.0
19693,KALLANG/WHAMPOA,3 ROOM,Terrace,249.0,1053888.0
92512,BISHAN,EXECUTIVE,Maisonette,243.0,1092888.0
94959,BISHAN,EXECUTIVE,Maisonette,243.0,1001000.0
107851,KALLANG/WHAMPOA,3 ROOM,Terrace,241.0,1235000.0
35797,KALLANG/WHAMPOA,3 ROOM,Terrace,237.0,1185000.0
95808,KALLANG/WHAMPOA,3 ROOM,Terrace,222.0,1100000.0
8868,KALLANG/WHAMPOA,3 ROOM,Terrace,215.0,830000.0
13950,CHOA CHU KANG,EXECUTIVE,Premium Maisonette,215.0,888000.0
20828,CHOA CHU KANG,EXECUTIVE,Premium Maisonette,215.0,900000.0


In [19]:
df["resale_price"].describe()

count    2.393300e+05
mean     5.342959e+05
std      1.919414e+05
min      1.400000e+05
25%      3.900000e+05
50%      5.020000e+05
75%      6.400000e+05
max      1.728000e+06
Name: resale_price, dtype: float64

In [20]:
df.nlargest(
    10,
    "resale_price"
)[
    ["month", "town", "flat_type",
     "floor_area_sqm", "storey_range",
     "resale_price"]
]

,month,town,flat_type,floor_area_sqm,storey_range,resale_price
225392,2026-04-01,BUKIT MERAH,5 ROOM,113.0,46 TO 48,1728000.0
232726,2026-02-01,QUEENSTOWN,5 ROOM,122.0,19 TO 21,1700000.0
225363,2026-08-01,BUKIT MERAH,5 ROOM,112.0,28 TO 30,1688888.0
212074,2025-06-01,QUEENSTOWN,5 ROOM,122.0,22 TO 24,1658888.0
223809,2026-08-01,BISHAN,EXECUTIVE,162.0,22 TO 24,1650000.0
232730,2026-06-01,QUEENSTOWN,5 ROOM,122.0,04 TO 06,1650000.0
225360,2026-03-01,BUKIT MERAH,5 ROOM,112.0,25 TO 27,1648888.0
199476,2025-11-01,BISHAN,5 ROOM,120.0,34 TO 36,1632000.0
226178,2026-05-01,CENTRAL AREA,5 ROOM,105.0,43 TO 45,1630000.0
199498,2025-11-01,BISHAN,EXECUTIVE,163.0,22 TO 24,1600000.0


In [6]:
df["lease_commence_date"].describe()

count    239330.000000
mean       1996.608858
std          14.387890
min        1966.000000
25%        1985.000000
50%        1997.000000
75%        2012.000000
max        2023.000000
Name: lease_commence_date, dtype: float64

In [7]:
df["lease_commence_date"].min(), df["lease_commence_date"].max()

(np.int64(1966), np.int64(2023))

### 7. Check impossible numerical values

In [23]:
(df["floor_area_sqm"] <= 0).sum()

np.int64(0)

In [24]:
(df["resale_price"] <= 0).sum()

np.int64(0)

In [25]:
(df["lease_commence_date"] > df["month"].dt.year).sum()

np.int64(0)

### 8. Final data-quality check

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239330 entries, 0 to 239329
Data columns (total 14 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   month                         239330 non-null  datetime64[ns]
 1   town                          239330 non-null  object        
 2   flat_type                     239330 non-null  object        
 3   block                         239330 non-null  object        
 4   street_name                   239330 non-null  object        
 5   storey_range                  239330 non-null  object        
 6   floor_area_sqm                239330 non-null  float64       
 7   flat_model                    239330 non-null  object        
 8   lease_commence_date           239330 non-null  int64         
 9   remaining_lease               239330 non-null  object        
 10  resale_price                  239330 non-null  float64       
 11  remaining_lea

In [27]:
df.isna().sum()

month                           0
town                            0
flat_type                       0
block                           0
street_name                     0
storey_range                    0
floor_area_sqm                  0
flat_model                      0
lease_commence_date             0
remaining_lease                 0
resale_price                    0
remaining_lease_years           0
remaining_lease_extra_months    0
remaining_lease_months          0
dtype: int64

In [28]:
df.shape

(239330, 14)

# Feature Engineering

Create analysis-ready variables from the original HDB transaction fields.

## 1. Transaction Year

In [14]:
df["year"] = df["month"].dt.year

In [15]:
df[["month", "year"]].head()

,month,year
0,2017-01-01,2017
1,2017-01-01,2017
2,2017-01-01,2017
3,2017-01-01,2017
4,2017-01-01,2017


## 2. Transaction Quarter

In [16]:
df["quarter"] = df["month"].dt.to_period("Q")

In [17]:
df[["month", "year", "quarter"]].head()

,month,year,quarter
0,2017-01-01,2017,2017Q1
1,2017-01-01,2017,2017Q1
2,2017-01-01,2017,2017Q1
3,2017-01-01,2017,2017Q1
4,2017-01-01,2017,2017Q1


In [18]:
df["quarter"].unique()[:10]

<PeriodArray>
['2017Q1', '2017Q2', '2017Q3', '2017Q4', '2018Q1', '2018Q2', '2018Q3',
 '2018Q4', '2019Q1', '2019Q2']
Length: 10, dtype: period[Q-DEC]

## 3 Price per Square Metre

Standardize transaction prices by floor area to improve comparisons across properties of different sizes.

In [19]:
df["price_per_sqm"] = (
    df["resale_price"] / df["floor_area_sqm"]
)

In [20]:
df[
    ["resale_price", "floor_area_sqm", "price_per_sqm"]
].head()

,resale_price,floor_area_sqm,price_per_sqm
0,232000.0,44.0,5272.727273
1,250000.0,67.0,3731.343284
2,262000.0,67.0,3910.447761
3,265000.0,68.0,3897.058824
4,265000.0,67.0,3955.223881


In [21]:
df["price_per_sqm"] = df["price_per_sqm"].round(2)

In [22]:
df["price_per_sqm"].describe()

count    239330.000000
mean       5587.991914
std        1675.557732
min        2089.550000
25%        4404.760000
50%        5307.690000
75%        6344.830000
max       16148.940000
Name: price_per_sqm, dtype: float64

## 4. Approximate Flat Age

Estimate flat age at the time of transaction using transaction year minus lease commencement year.

In [23]:
df["flat_age"] = (
    df["year"] - df["lease_commence_date"]
)

In [24]:
df[
    ["year", "lease_commence_date", "flat_age"]
].head(10)

,year,lease_commence_date,flat_age
0,2017,1979,38
1,2017,1978,39
2,2017,1980,37
3,2017,1980,37
4,2017,1980,37
5,2017,1981,36
6,2017,1979,38
7,2017,1976,41
8,2017,1979,38
9,2017,1979,38


In [25]:
df["flat_age"].describe()

count    239330.000000
mean         24.950850
std          14.293738
min           1.000000
25%          11.000000
50%          25.000000
75%          37.000000
max          60.000000
Name: flat_age, dtype: float64

In [26]:
(df["flat_age"] < 0).sum()

np.int64(0)

## 5 Storey Midpoint

Convert categorical storey ranges into their numerical midpoint while retaining the original storey range.

In [27]:
storey_split = df["storey_range"].str.split(" TO ", expand=True)

df["storey_lower"] = storey_split[0].astype(int)
df["storey_upper"] = storey_split[1].astype(int)

df["storey_midpoint"] = (
    df["storey_lower"] + df["storey_upper"]
) / 2

In [28]:
df[
    ["storey_range", "storey_lower", "storey_upper", "storey_midpoint"]
].drop_duplicates().sort_values("storey_midpoint")

,storey_range,storey_lower,storey_upper,storey_midpoint
1,01 TO 03,1,3,2.0
3,04 TO 06,4,6,5.0
13,07 TO 09,7,9,8.0
0,10 TO 12,10,12,11.0
46,13 TO 15,13,15,14.0
116,16 TO 18,16,18,17.0
47,19 TO 21,19,21,20.0
48,22 TO 24,22,24,23.0
506,25 TO 27,25,27,26.0
241,28 TO 30,28,30,29.0


## 6. Remaining Lease (Years)

Convert total remaining lease months into a continuous year measure for easier interpretation.

In [34]:
df["remaining_lease_years_numeric"] = (
    df["remaining_lease_months"] / 12
).round(2)

In [35]:
df[
    [
        "remaining_lease",
        "remaining_lease_months",
        "remaining_lease_years_numeric"
    ]
].head(10)

,remaining_lease,remaining_lease_months,remaining_lease_years_numeric
0,61 years 04 months,736,61.33
1,60 years 07 months,727,60.58
2,62 years 05 months,749,62.42
3,62 years 01 month,745,62.08
4,62 years 05 months,749,62.42
5,63 years,756,63.00
6,61 years 06 months,738,61.50
7,58 years 04 months,700,58.33
8,61 years 06 months,738,61.50
9,61 years 04 months,736,61.33


## 7. Feature Validation

In [36]:
df[
    [
        "month",
        "year",
        "quarter",
        "resale_price",
        "floor_area_sqm",
        "price_per_sqm",
        "flat_age",
        "storey_midpoint",
        "remaining_lease_years_numeric"
    ]
].head(10)

,month,year,quarter,resale_price,floor_area_sqm,price_per_sqm,flat_age,storey_midpoint,remaining_lease_years_numeric
0,2017-01-01,2017,2017Q1,232000.0,44.0,5272.73,38,11.0,61.33
1,2017-01-01,2017,2017Q1,250000.0,67.0,3731.34,39,2.0,60.58
2,2017-01-01,2017,2017Q1,262000.0,67.0,3910.45,37,2.0,62.42
3,2017-01-01,2017,2017Q1,265000.0,68.0,3897.06,37,5.0,62.08
4,2017-01-01,2017,2017Q1,265000.0,67.0,3955.22,37,2.0,62.42
5,2017-01-01,2017,2017Q1,275000.0,68.0,4044.12,36,2.0,63.00
6,2017-01-01,2017,2017Q1,280000.0,68.0,4117.65,38,5.0,61.50
7,2017-01-01,2017,2017Q1,285000.0,67.0,4253.73,41,5.0,58.33
8,2017-01-01,2017,2017Q1,285000.0,68.0,4191.18,38,5.0,61.50
9,2017-01-01,2017,2017Q1,285000.0,67.0,4253.73,38,2.0,61.33


In [37]:
df[
    [
        "price_per_sqm",
        "flat_age",
        "storey_midpoint",
        "remaining_lease_years_numeric"
    ]
].describe()

,price_per_sqm,flat_age,storey_midpoint,remaining_lease_years_numeric
count,239330.000000,239330.000000,239330.000000,239330.000000
mean,5587.991914,24.950850,8.788288,74.092959
std,1675.557732,14.293738,5.955835,14.324530
min,2089.550000,1.000000,2.000000,39.330000
25%,4404.760000,11.000000,5.000000,62.170000
50%,5307.690000,25.000000,8.000000,73.830000
75%,6344.830000,37.000000,11.000000,88.580000
max,16148.940000,60.000000,50.000000,97.750000


In [38]:
print("Negative flat ages:", (df["flat_age"] < 0).sum())
print("Non-positive price/sqm:", (df["price_per_sqm"] <= 0).sum())
print("Non-positive remaining lease:", (df["remaining_lease_months"] <= 0).sum())

Negative flat ages: 0
Non-positive price/sqm: 0
Non-positive remaining lease: 0


## 8. Export Processed Dataset

In [ ]:
df = df[['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'year', 'quarter', 'price_per_sqm', 'flat_age', 'storey_lower', 'storey_upper', 'storey_midpoint', 'remaining_lease_years', 'remaining_lease_extra_months', 'remaining_lease_months', 'remaining_lease_years_numeric']]
df.to_csv(
    PROJECT_ROOT / "data" / "processed" / "hdb_resale_clean.csv",
    index=False
)